In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok gradio timm google-generativeai nest-asyncio Pillow

In [ ]:
import os
import json
import asyncio
import threading
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import timm
from torchvision import transforms
import google.generativeai as genai
import gradio as gr
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import io
import requests

nest_asyncio.apply()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Using device: cuda


# Mount Google Drive & Load Models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted!


In [ ]:
DRIVE_MODEL_PATH = '/content/drive/MyDrive/skin_disease_model/best_model.pt'
DRIVE_CLASS_PATH = '/content/drive/MyDrive/skin_disease_model/class_names.json'

with open(DRIVE_CLASS_PATH, 'r') as f:
    class_info = json.load(f)

CLASS_NAMES = class_info['class_names']
CLEAN_NAMES = class_info['clean_names']
NUM_CLASSES = class_info['num_classes']

print(f'Loaded {NUM_CLASSES} classes:')
for raw, clean in CLEAN_NAMES.items():
    print(f'  {clean}')

Loaded 10 classes:
  Eczema
  Warts & Viral Infections
  Melanoma
  Atopic Dermatitis
  Basal Cell Carcinoma
  Melanocytic Nevi
  Benign Keratosis
  Psoriasis & Lichen Planus
  Seborrheic Keratoses
  Tinea & Fungal Infections


In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)

checkpoint = torch.load(DRIVE_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f'Model loaded successfully!')
print(f'Best Val Accuracy from training: {checkpoint["val_acc"]:.2f}%')

Model loaded successfully!
Best Val Accuracy from training: 84.24%


# Image Preprocessing & Inference Pipeline

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def predict_disease(image: Image.Image):
    """
    Takes a PIL image, returns (disease_name, confidence)
    """
    img_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = probabilities.max(1)

    raw_class = CLASS_NAMES[predicted_idx.item()]
    clean_class = CLEAN_NAMES.get(raw_class, raw_class)
    confidence_score = round(confidence.item(), 4)

    top3_probs, top3_indices = probabilities[0].topk(3)
    top3 = [
        {
            'disease': CLEAN_NAMES.get(CLASS_NAMES[idx.item()], CLASS_NAMES[idx.item()]),
            'confidence': round(prob.item(), 4)
        }
        for prob, idx in zip(top3_probs, top3_indices)
    ]

    return clean_class, confidence_score, top3


print('Inference pipeline ready!')

Inference pipeline ready!


# Gemini LLM Integration

In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 12.8 MB/s eta 0:00:00


In [ ]:
from groq import Groq

GROQ_API_KEY = '***************************'

groq_client = Groq(api_key=GROQ_API_KEY)


def get_llm_recommendations(disease: str, confidence: float):
    prompt = f"""
You are a helpful medical assistant. A skin disease detection AI has analyzed a patient's skin image.

Detection Result:
- Disease: {disease}
- Confidence: {confidence * 100:.1f}%

Please provide a structured response in the following JSON format ONLY, no extra text:
{{
  "recommendations": "2-3 sentences about what this disease is and general medical advice",
  "next_steps": "2-3 specific action steps the patient should take",
  "tips": "2-3 practical daily tips to manage or prevent worsening",
  "urgency": "low or medium or high"
}}

Important: Always recommend consulting a dermatologist for proper diagnosis.
"""
    try:
        response = groq_client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.7,
            max_tokens=500
        )
        text = response.choices[0].message.content.strip()
        text = text.replace('```json', '').replace('```', '').strip()
        result = json.loads(text)
        return result

    except Exception as e:
        print(f'LLM Error: {e}')
        return {
            'recommendations': f'{disease} detected. Please consult a dermatologist.',
            'next_steps': 'Schedule an appointment with a dermatologist as soon as possible.',
            'tips': 'Avoid scratching the affected area. Keep the skin clean and moisturized.',
            'urgency': 'medium'
        }


test = get_llm_recommendations('Eczema', 0.92)
print('LLM test response:')
print(json.dumps(test, indent=2))

LLM test response:
{
  "recommendations": "Eczema is a chronic skin condition characterized by dry, itchy, and inflamed skin, and while the AI detection result suggests a high confidence level, it's essential to consult a dermatologist for a proper diagnosis and personalized treatment plan. A dermatologist can help determine the best course of treatment, which may include topical creams, oral medications, or lifestyle changes. It's crucial to seek professional advice to manage the condition effectively.",
  "next_steps": "Schedule an appointment with a dermatologist as soon as possible to confirm the diagnosis and discuss treatment options. Ask the dermatologist about any necessary tests or examinations to determine the severity of the condition. Keep a record of symptoms and any factors that may trigger or worsen the condition to share with the dermatologist.",
  "tips": "Keep the skin moisturized with gentle, fragrance-free creams or ointments to reduce dryness and itching. Avoid scr

# FastAPI Backend

In [ ]:
app = FastAPI(
    title='Skin Disease Detection API',
    description='Upload a skin image to detect disease and get LLM recommendations',
    version='1.0.0'
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)

class QuickAnalysis(BaseModel):
    disease: str
    confidence: float

class FullAnalysis(BaseModel):
    disease: str
    confidence: float
    recommendations: str
    next_steps: str
    tips: str
    urgency: str
    top3_predictions: list


@app.get('/')
def root():
    return {'message': 'Skin Disease Detection API is running!'}


@app.post('/analyze_skin', response_model=QuickAnalysis)
async def analyze_skin_quick(file: UploadFile = File(...)):
    """
    Quick analysis - returns disease and confidence only
    """
    if not file.content_type.startswith('image/'):
        raise HTTPException(status_code=400, detail='File must be an image')

    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert('RGB')

    disease, confidence, top3 = predict_disease(image)

    return {
        'disease': disease,
        'confidence': confidence
    }


@app.post('/analyze_skin_full', response_model=FullAnalysis)
async def analyze_skin_full(file: UploadFile = File(...)):
    """
    Full analysis - returns disease + confidence + LLM recommendations
    """
    if not file.content_type.startswith('image/'):
        raise HTTPException(status_code=400, detail='File must be an image')

    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert('RGB')

    disease, confidence, top3 = predict_disease(image)

    llm_result = get_llm_recommendations(disease, confidence)

    return {
        'disease': disease,
        'confidence': confidence,
        'recommendations': llm_result.get('recommendations', ''),
        'next_steps': llm_result.get('next_steps', ''),
        'tips': llm_result.get('tips', ''),
        'urgency': llm_result.get('urgency', 'medium'),
        'top3_predictions': top3
    }


@app.get('/health')
def health_check():
    return {
        'status': 'healthy',
        'model': 'EfficientNet-B0',
        'classes': NUM_CLASSES,
        'device': str(device)
    }


print('FastAPI app defined!')

FastAPI app defined!


# Start FastAPI with ngrok

In [ ]:
from pyngrok import ngrok
import time

NGROK_AUTH_TOKEN = '****************************'

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def run_fastapi():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

thread = threading.Thread(target=run_fastapi, daemon=True)
thread.start()
time.sleep(2)

public_url = ngrok.connect(8000)
API_URL = str(public_url).replace('NgrokTunnel: "', '').replace('" -> "http://localhost:8000"', '')

print(f'FastAPI is running!')
print(f'Public API URL: {API_URL}')
print(f'API Docs: {API_URL}/docs')
print(f'Health Check: {API_URL}/health')

FastAPI is running!
Public API URL: https://nonecstatically-tabulable-jovanni.ngrok-free.dev
API Docs: https://nonecstatically-tabulable-jovanni.ngrok-free.dev/docs
Health Check: https://nonecstatically-tabulable-jovanni.ngrok-free.dev/health


# Gradio UI

In [ ]:
def analyze_image(image):
    """
    Gradio function - takes image, calls FastAPI, returns results
    """
    if image is None:
        return 'Please upload an image', '', '', '', ''

    try:
        pil_image = Image.fromarray(image.astype('uint8'), 'RGB')
        img_bytes = io.BytesIO()
        pil_image.save(img_bytes, format='JPEG')
        img_bytes.seek(0)

        response = requests.post(
            f'{API_URL}/analyze_skin_full',
            files={'file': ('image.jpg', img_bytes, 'image/jpeg')},
            timeout=30
        )
        result = response.json()

        disease     = f"{result['disease']} ({result['confidence']*100:.1f}% confidence)"
        urgency     = f"⚠️ Urgency Level: {result['urgency'].upper()}"
        recs        = result['recommendations']
        next_steps  = result['next_steps']
        tips        = result['tips']

        top3_text = 'Top 3 Predictions:\n'
        for i, pred in enumerate(result['top3_predictions'], 1):
            top3_text += f"{i}. {pred['disease']}: {pred['confidence']*100:.1f}%\n"

        return disease, urgency, recs, next_steps, tips + '\n\n' + top3_text

    except Exception as e:
        return f'Error: {str(e)}', '', '', '', ''


with gr.Blocks(title='Skin Disease Detector', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🩺 Skin Disease Detection & AI Advisor
    Upload a skin image to detect the disease and get AI-powered recommendations.
    > ⚠️ **Disclaimer:** This tool is for educational purposes only. Always consult a dermatologist.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label='Upload Skin Image', type='numpy')
            analyze_btn = gr.Button('🔍 Analyze', variant='primary', size='lg')

        with gr.Column(scale=2):
            disease_output  = gr.Textbox(label='🦠 Detected Disease', interactive=False)
            urgency_output  = gr.Textbox(label='🚨 Urgency Level',    interactive=False)
            recs_output     = gr.Textbox(label='📋 Recommendations',  interactive=False, lines=3)
            steps_output    = gr.Textbox(label='👣 Next Steps',       interactive=False, lines=3)
            tips_output     = gr.Textbox(label='💡 Tips & Top Predictions', interactive=False, lines=5)

    analyze_btn.click(
        fn=analyze_image,
        inputs=[image_input],
        outputs=[disease_output, urgency_output, recs_output, steps_output, tips_output]
    )

    gr.Markdown("""
    ### Detectable Diseases:
    Eczema | Melanoma | Atopic Dermatitis | Basal Cell Carcinoma | Melanocytic Nevi |
    Benign Keratosis | Psoriasis & Lichen Planus | Seborrheic Keratoses | Tinea & Fungal Infections | Warts & Viral Infections
    """)

print('Gradio UI defined!')

/tmp/ipykernel_3251/3962066632.py:42: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Skin Disease Detector', theme=gr.themes.Soft()) as demo:


Gradio UI defined!


In [ ]:
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e18e0f80c6999dca0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
